# Text Feature Extraction — CLIP + Long-CLIP

- CLIP ViT-B/32 (77 tokens): title | llm_enhanced_text
- Long-CLIP ViT-L/14 (248 tokens): title | llm_enhanced_text | description

输出均为 512-dim, 对齐 image / ID embeddings.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch, pandas as pd, numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel
from sklearn.decomposition import PCA

META_CSV        = "final/item_meta_llm_enhanced.csv"
OUTPUT_DIR      = "final/features"
CLIP_MODEL      = "openai/clip-vit-base-patch32"
LONG_CLIP_MODEL = "zer0int/LongCLIP-GmP-ViT-L-14"
BATCH_SIZE      = 64
FEATURE_DIM     = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [2]:
df = pd.read_csv(META_CSV)
print(f"Items: {len(df)}")
enhanced_ok = (df["llm_enhanced_text"].notna() & (df["llm_enhanced_text"]!="")).sum()
print(f"LLM enhanced: {enhanced_ok}/{len(df)}")

df["clip_text"] = (
    df["title"].fillna("") + " | " + df["llm_enhanced_text"].fillna("")
)
df["longclip_text"] = (
    df["title"].fillna("") + " | " +
    df["llm_enhanced_text"].fillna("") + " | " +
    df["description"].fillna("").str.slice(0, 600)
)

print(f"CLIP text avg len: {df['clip_text'].str.len().mean():.0f}")
print(f"Long-CLIP text avg len: {df['longclip_text'].str.len().mean():.0f}")

Items: 43528
LLM enhanced: 43528/43528
CLIP text avg len: 334
Long-CLIP text avg len: 835


In [ ]:
# Dataset and Extract
class TextDataset(Dataset):
    def __init__(self, texts, item_ids):
        self.texts = texts
        self.ids = item_ids
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.ids[idx]

def extract(model, processor, dataloader, max_len, device, desc):
    model.eval()
    feats, ids = [], []
    with torch.no_grad():
        for batch_texts, batch_ids in tqdm(dataloader, desc=desc):
            inp = processor(text=batch_texts, return_tensors="pt",
                           padding=True, truncation=True,
                           max_length=max_len).to(device)
            bf = model.get_text_features(**inp).cpu().numpy()
            feats.append(bf)
            ids.extend(int(x) for x in batch_ids)
    return np.concatenate(feats, axis=0), ids

## 1. CLIP ViT-B/32 (77 tokens)

In [4]:
print("Loading CLIP ViT-B/32...")
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(device).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_MODEL)

texts = df["clip_text"].tolist()
item_ids = df["parent_asin"].tolist()

ds = TextDataset(texts, item_ids)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

clip_feats, clip_ids = extract(clip_model, clip_proc, loader, 77, device, "CLIP (77)")

np.save(f"{OUTPUT_DIR}/text_features_clip_512.npy", clip_feats)
pd.DataFrame({"item_id": clip_ids}).to_csv(f"{OUTPUT_DIR}/text_id_map_clip.csv", index=False)
print(f"CLIP done: {clip_feats.shape}")

del clip_model, clip_proc
torch.cuda.empty_cache()

Loading CLIP ViT-B/32...


d:\Python2\Python311\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
CLIP (77): 100%|██████████| 681/681 [01:21<00:00,  8.34it/s]


CLIP done: (43528, 512)


## 2. Long-CLIP ViT-L/14 (248 tokens → truncate first 512 dims)

PCA 会破坏余弦结构，直接用前 512 维截断保留信号。

In [5]:
print(f"Loading Long-CLIP: {LONG_CLIP_MODEL}")
lclip_model = CLIPModel.from_pretrained(LONG_CLIP_MODEL).to(device).eval()
lclip_proc = CLIPProcessor.from_pretrained(LONG_CLIP_MODEL)

texts = df["longclip_text"].tolist()
ds = TextDataset(texts, item_ids)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

lclip_raw, lclip_ids = extract(lclip_model, lclip_proc, loader, 248, device, "Long-CLIP (248)")
print(f"Long-CLIP raw dim: {lclip_raw.shape[1]}")

Loading Long-CLIP: zer0int/LongCLIP-GmP-ViT-L-14


Long-CLIP (248): 100%|██████████| 681/681 [09:30<00:00,  1.19it/s]

Long-CLIP raw dim: 768


In [ ]:
# Truncate to 512
# 不用 PCA（会破坏余弦相似度结构），直接取前 512 维
lclip_aligned = lclip_raw[:, :FEATURE_DIM]

np.save(f"{OUTPUT_DIR}/text_features_longclip_512.npy", lclip_aligned)
pd.DataFrame({"item_id": lclip_ids}).to_csv(f"{OUTPUT_DIR}/text_id_map_longclip.csv", index=False)
print(f"Long-CLIP done: {lclip_aligned.shape} (first 512 of {lclip_raw.shape[1]})")

In [ ]:
print("="*50)
print("All text features saved to final/features/")
print(f"  text_features_clip_512.npy      -> {clip_feats.shape}")
print(f"  text_features_longclip_512.npy  -> {lclip_aligned.shape}")

All text features saved to final/features/
  text_features_clip_512.npy      -> (43528, 512)
  text_features_longclip_512.npy  -> (43528, 512)
